# Õppetund 18: AI agentide turvamine krüptograafiliste kviitungitega

## Praktiline märkmik

See märkmik juhendab läbi nelja harjutuse:

1. **Allkirjasta oma esimene kviitung** agendi tööriista kutsumise kohta ja kinnita see.
2. **Muuda kviitungit** ja vaata, kuidas kinnitamine ebaõnnestub.
3. **Ehitada kolmekihtne kviitungiahel** ja kinnitada ahela terviklikkus.
4. **Pakkida Microsofti Agent raamistiku tööriistakutse** nii, et iga tegevus genereerib kviitungi.

Kõik krüptograafilised põhioperatsioonid on imporditud hästi hooldatud teekidest (`pynacl` Ed25519 jaoks, `jcs` RFC 8785 kanonilise JSON jaoks, `hashlib` Pythoni standardteegist SHA-256 jaoks). Kviitungi loogika ise on puhas Python, mida saad lugeda ja muuta.

Käivita lahtrid järjekorras. Iga sektsioon on lühike ja iseseisev.


## Häälestus

Paigalda kaks sõltuvust. Mõlemal on liberaalsed litsentsid (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Abivahendid

Need kaks abihoidjat tegelevad base64url kodeerimisega (ilma täiendamiseta) ja suvaliste objektide SHA-256 räsi genereerimisega. Need hoiavad ülejäänud märkmeraamatu keskendunud vastuavaldise loogikale.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sektsioon 1: Allkirjasta oma esimene kviitung

Kujutame ette, et meie agendil **Contoso Travel** on just klienti silmas pidades otsitud lende Sydneyst Los Angelesse. Soovime salvestada gateway avalduse selle tööriista väljakutse kohta, et tulevane audiitor saaks selle eraldi kinnitatud väljastaja võtit kasutades kinnitada.

### Samm 1.1: Genereeri allkirjastamisvõti

Tootmiskeskkonnas elaks agendi allkirjastamisvõti riistvaralises turbemoodulis (HSM), Azure Key Vault'is või sarnases kaitstud andmehoidlas. Sellele õppetunnile genereerime mälus uue võtme.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Samm 1.2: Koosta kviitungi andmepakett

Andmepakett sisaldab kõike seda, mida soovime kviitungis tõendada: kes tegutses, millist tööriista kasutas, milliste argumentidega, mis tagastati, millise poliitika alusel ja millal. Me räsiime argumendid ja tulemuse, selle asemel et neid otse lisada, nii et kviitung ei lekita tundlikku sisu.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Samm 1.3: Allkirjasta ja koosta kviitung

Kolm sammu:

1. Normaliseeri andmed JCS abil nii, et kaks sama loogilise kviitungi genereerivat rakendust annaksid täpselt samad baitide jada.
2. Allkirjasta normaliseeritud baitid otse Ed25519 privaatvõtmega. PureEdDSA räsi sõnumi sees, seega täiendav eelhäälestus muudaks protokolli.

Allkiri lisatakse seejärel algsetele andmetele, et saada lõplik kviitung.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Samm 1.4: Tõendi kontrollimine

Kontrollimine pöörab protsessi ümber. Me eemaldame allkirja, arvutame kanonilised baidid uuesti ja kontrollime allkirja väljaandja avaliku võtmega.

Kontrollimine toimub võrguühenduseta: pole teenust, mida kutsuda, ega võrguühendust. Selleks on siiski vaja üht lisaks tõendile: väljaandja avalikku võtit, mis saadakse kanali kaudu, mida audiitor juba usaldab, näiteks väljaandja avaldatud JWK komplekt või eelnevalt fikseeritud väärtus. Väljade `public_key` sees tõendis ei saa seda rolli täita, sest see, kes tõendi kirjutab, kirjutab selle välja ka ise. Lõik 2 näitab selle takistatud võltsimise juhtumit.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Sa peaksid nägema `Receipt is valid: True`, `Pre-hashed receipt valid: False` ja `Malformed receipts rejected: True`. Positiivne test näitab, et otse-JCS-i allkirjastamise rada toimib; negatiivne kontroll muudab allkirja-ulatuse reegli täidetavaks selle asemel, et jääda lihtsalt kirjeldavaks; viimane rida näitab valesti vormistatud sisendi tagastamist `False` asemel, et tekitada erand.  


## Sektsioon 2: Kviitungi võltsimine

Kviitungide kogu mõte on selles, et nende võltsimine on selgesti nähtav. Proovime seda tõestada.

Muudame täpselt ühe tähemärgi kviitungil ja vaatame, kuidas kontroll ebaõnnestub.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Mis just juhtus?

Kui me muutsime `policy_id`, muutusid ka kanonilised baitid. Allkiri (mis oli algsete kanoniliste baitide peal) ei klapi enam. Kontrollimine tagastab õigesti `False`.

Pole võimalik muuta allkirjastatud välja, mis tähendab kõike väljaspool `signature` objekti, ja siiski lasta kviitungil väljaandja avaliku võtmega kontrollimisel läbi minna, kui ründajal pole väljaandja privaatvõtit. `signature` objekt ise ei ole allkirjastatud: `sig` on allkiri ja `verify_receipt` ignoreerib `public_key`, seega selle välja muutmine iseenesest midagi ei muuda. Niikaua kui privaatvõti on võtmekogus ja kontrollijad kasutavad väljaandja iseseisvalt avaldatud võtit, on allkirjastatud välja muutmine võimatu peita. See tingimus on oluline: järgmine lahter näitab muudetud kviitungit, mis kontrollib täiuslikult erineva võtme vastu.

Proovi ise: muuda eelpool lahtris `tool_name` või `agent_id` või `timestamp` ja käivita uuesti. Iga muutus allkirjastatud väljas tekitab kehtetu kviitungi.


### Pahatahtlik muutmine, mida allkiri ei paljasta

Ülaltoodud ründaja ei uuendanud allkirja. Hoolikam ründaja teeks nii: muudaks kviitungi,
allkirjastaks selle võtmega, mille on ise genereerinud, ja paneks selle võtme
`public_key` väljale. Tulemus oleks kviitung, mille allkiri on täiesti ehtne
selle sisu jaoks.

Käivitage see ja vaadake, kumb kahest kontrollist selle kinni püüab.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Sa peaksid nägema `True` ja siis `False`. Kolm rida `assert` teevad sellest
kontrolli: lahter viskab vea, kui võltsing kunagi väljastaja võtme vastu tõestub.


Võltsing on sisemiselt järjepidev. Selle allkiri vastab tõepoolest selle
sisule ja võtmele, mida see kannab, nii et tõendaja, kes võtme kviitungist välja loeb,
tunnistab selle kehtivaks. Kviitungi kohta ei tundu midagi valesti, sest
midagi selles ei ole inkonsistentne.

Ainsaks, mis selle kinni püüab, on kontroll võtmega, millel ründajal polnud
võimalikku mõjutada. Seetõttu võtab `verify_receipt` võtme argumendina: funktsiooni
signatuuriga `verify_receipt(receipt)` ei saa turvaliselt kirjutada,
sest ainus võtmevariant, mida see võiks kasutada, on see, mille ründaja andis.

Sama kviitung salvestatakse kui `sample_receipts/04_resigned_receipt.json`, kui soovid seda väljaspool märkmikku
vaadata.


## Sektsioon 3: Keti tõendite ühendamine

Üks tõend kaitseb üht toimingut. Enamik agente teeb palju toiminguid. Selleks, et kogu järjestus oleks näha, kui seda on püüdsud muuta, ühendame iga tõendi eelmisega, lisades uue tõendi andmepaketti eelmise tõendi räsi.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Kui keegi eemaldab või muudab tõendi järjekorda kujutletavas järjestuses, katkeb järgmine link. Lõpetamise või allkirjastaja juhtimisega järelliide ümberkirjutuse tuvastamiseks on samuti vajalik kinnitatud praegune pea, väline kontrollpunkt või autentitud elanikkonna kohustus.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Nüüd katke ahel, võltsides keskmist kviitungit ja kontrollides uuesti. Võltsitud kviitung ebaõnnestub oma allkirjakontrollis, JA järgmine kviitung ebaõnnestub oma ahela-linkide kontrollis (sest selle `previous_receipt_hash` ei vasta enam muudetud keskmise kviitungi räsidele).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Kviitung 0 kontrollib endiselt õigesti (seda ei muudetud ja tal puudub eelkäija, millele toetuda). Kviitung 1 ebaõnnestub oma allkirja kontroll, kuna me muutsime `tool_args_hash`. Kviitung 2 ebaõnnestub oma ahela lingi kontroll, sest selle `previous_receipt_hash` arvutati originaalse (nüüd muudetud) kviitungi 1 vastu.

Isegi kui ründaja kirjutab muudetud kviitungi 1 uuesti allkirjastatud, siis uus allkiri ebaõnnestub emissari võtme vastas, välja arvatud juhul, kui tal on emissari privaatvõti (oma võtmega uuesti allkirjastamine ebaõnnestub, nagu lõigus 2 näidati), ja kviitungi 2 ahela lingi vastavuse puudumine paljastaks endiselt muutmise. Muudatuse varjamiseks peaks ründaja kirjutama uuesti allkirja igale kviitungile muudatuspunktist edasi, mis nõuab emissari privaatvõtme omamist.


## 4. jaotis: Agenttööriista kutsumise ümbermähkimine kviitungi allkirjastamisega

Tõelises juurutuses ei taha te, et iga agendi autor meenutaks `make_receipt` kutsumist. Soovite, et kviitungi allkirjastamine oleks automaatne iga tööriista käivituse korral.

Siin on lihtsaim mustritüüp: ümbrisklass, mis võtab vastu mis tahes kutsutava tööriistafunktsiooni ja tagastab selle kviitungit väljastava versiooni. See kohandub mistahes agendifraamistikuga, sealhulgas Microsoft Agent Frameworkiga (`agent_framework.foundry`).

Kui teil pole seadistatud Microsoft Foundry projekti, näitab allolev kohalik simulatsioon siiski seda mustrit.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integreerimine Microsofti agendi raamistikuga

Ülaltoodud `ReceiptedTool` ümbris on raamistikust sõltumatu. Selle kasutamiseks Microsofti agendi raamistikku ehitatud agendi sees registreerige ümbritsetud funktsioon tööriistana. Joonis (te asendaksite võltsimise päris Microsoft Foundry tööriista registreerimisega):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agendi raamistik ei pea teadma midagi tšekkidest. Tšeki allkirjastamine on mähitud tööriista ümber, mitte raamistikku sisse keeratud. See on viis, kuidas lisada päritolu olemasolevale agendi koodile ilma agendi ümberkirjutamiseta.


## Kokkuvõte ja täiendav väljakutse

Sul on:

- Genereeritud Ed25519 võtmepar.
- Koostatud ja allkirjastatud agenttööriista kõne kviitung.
- Kviitung kontrollitud võrguühenduseta, kasutades üksnes väljaandja avalikku võtit, mis on saadud kviitungist sõltumatult.
- Muudetud kviitungit ja täheldatud kontrollimisel ebaõnnestumist.
- Koostatud hash-ahelaks seotud kolmest kviitungist koosnev järjestus.
- Muudetud ahela keskmist osa ja täheldatud nii allkirja kui ka ahela lingi ebaõnnestumist.
- Ümbritsetud tööriista funktsioon automaatse kviitungi allkirjastamisega.

**Täiendav väljakutse.** Laienda kviitungi skeemi `request_id` välja (UUID hajutatud jälgimiseks). Uuenda `make_receipt` nii, et see välja sisaldaks, ja kinnita, et kviitungid kontrollitakse ikka lõpust lõpuni. Seejärel muuda väljad pärast allkirjastamist ja kinnita, et kontrollimine ebaõnnestub. See sunnib sind mõistma, kuidas iga kaanoniline kodeeringu bait panustab allkirja.

**Oluline piir.** Püsiva võtme ja ahela konteksti puhul tõendavad kviitungid allkirja päritolu, sisu terviklikkust ja linki esitatud eellaaduri poole. Nad EI tõenda täitmist, allika tõde, usaldusväärset aega, täielikkust, poliitika hindamist ega õigsust. Kviitungid on alus. Valitsemine on süsteem, mida ehitad peale.

Loe õppetunni README-d uuesti, pidades seda piiri silmas. Kõige tavalisem viga, mida meeskonnad kviitungite puhul teevad, on arvata, et „meil on kviitungid“ tähendab „meid valitsetakse“. Seda see ei tähenda. Kviitungid teevad agendi käitumise auditeeritavaks. Nad ei muuda seda õigeks.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
